In [1]:
import re
from collections import defaultdict, Counter
import math

# -------------------------------
# 1. LOAD AND PREPROCESS CORPUS
# -------------------------------
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation
    tokens = text.split()
    return tokens

with open("corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

tokens = preprocess(text)

# Add start tokens for trigram
tokens = ['<s>', '<s>'] + tokens + ['</s>']

# -------------------------------
# 2. BUILD N-GRAM COUNTS
# -------------------------------
def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

unigrams = get_ngrams(tokens, 1)
bigrams = get_ngrams(tokens, 2)
trigrams = get_ngrams(tokens, 3)

unigram_counts = Counter(unigrams)
bigram_counts = Counter(bigrams)
trigram_counts = Counter(trigrams)

vocab = set(tokens)
V = len(vocab)

# -------------------------------
# 3. PROBABILITY FUNCTIONS
# -------------------------------

# Unigram probability
def unigram_prob(word):
    return unigram_counts[(word,)] / sum(unigram_counts.values())

# Bigram probability
def bigram_prob(w1, w2):
    return bigram_counts[(w1, w2)] / unigram_counts[(w1,)]

# Trigram probability
def trigram_prob(w1, w2, w3):
    return trigram_counts[(w1, w2, w3)] / bigram_counts[(w1, w2)]

# -------------------------------
# 4. LAPLACE SMOOTHING (ADD-1)
# -------------------------------
def laplace_bigram(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[(w1,)] + V)

def laplace_trigram(w1, w2, w3):
    return (trigram_counts[(w1, w2, w3)] + 1) / (bigram_counts[(w1, w2)] + V)

# -------------------------------
# 5. ADD-K SMOOTHING
# -------------------------------
def add_k_bigram(w1, w2, k):
    return (bigram_counts[(w1, w2)] + k) / (unigram_counts[(w1,)] + k * V)

def add_k_trigram(w1, w2, w3, k):
    return (trigram_counts[(w1, w2, w3)] + k) / (bigram_counts[(w1, w2)] + k * V)

# -------------------------------
# 6. BACKOFF MODEL
# -------------------------------
def backoff_prob(w1, w2, w3):
    # Try trigram
    if trigram_counts[(w1, w2, w3)] > 0:
        return trigram_prob(w1, w2, w3)
    # Backoff to bigram
    elif bigram_counts[(w2, w3)] > 0:
        return bigram_prob(w2, w3)
    # Backoff to unigram
    else:
        return unigram_prob(w3)

# -------------------------------
# 7. SENTENCE PROBABILITY (BACKOFF)
# -------------------------------
def sentence_probability(sentence):
    words = preprocess(sentence)
    words = ['<s>', '<s>'] + words + ['</s>']
    
    prob = 1.0
    for i in range(2, len(words)):
        p = backoff_prob(words[i-2], words[i-1], words[i])
        prob *= p
    
    return prob

# -------------------------------
# 8. DISPLAY COUNT TABLES
# -------------------------------
def print_top_counts(counter, name, n=10):
    print(f"\nTop {n} {name}:")
    for item, count in counter.most_common(n):
        print(item, ":", count)

print_top_counts(unigram_counts, "Unigrams")
print_top_counts(bigram_counts, "Bigrams")
print_top_counts(trigram_counts, "Trigrams")

# -------------------------------
# 9. TESTING
# -------------------------------
print("\n--- SAMPLE PROBABILITIES ---")
print("Laplace Bigram (tom, sawyer):", laplace_bigram("tom", "sawyer"))

print("Add-k Bigram k=0.1:", add_k_bigram("tom", "sawyer", 0.1))
print("Add-k Bigram k=0.5:", add_k_bigram("tom", "sawyer", 0.5))
print("Add-k Bigram k=0.7:", add_k_bigram("tom", "sawyer", 0.7))

sentence = "tom sawyer went home"
print("\nSentence:", sentence)
print("Backoff Probability:", sentence_probability(sentence))


Top 10 Unigrams:
('the',) : 3847
('and',) : 3052
('a',) : 1851
('to',) : 1783
('of',) : 1550
('he',) : 1182
('was',) : 1168
('it',) : 1125
('in',) : 991
('that',) : 903

Top 10 Bigrams:
('of', 'the') : 382
('in', 'the') : 320
('and', 'the') : 183
('it', 'was') : 180
('to', 'the') : 179
('he', 'was') : 147
('and', 'then') : 114
('there', 'was') : 113
('was', 'a') : 113
('he', 'had') : 113

Top 10 Trigrams:
('there', 'was', 'a') : 44
('the', 'project', 'gutenberg') : 30
('there', 'was', 'no') : 23
('out', 'of', 'the') : 22
('it', 'was', 'a') : 20
('by', 'and', 'by') : 20
('i', 'dont', 'know') : 19
('project', 'gutenberg', 'electronic') : 18
('he', 'did', 'not') : 17
('out', 'of', 'his') : 15

--- SAMPLE PROBABILITIES ---
Laplace Bigram (tom, sawyer): 0.002854633289415898
Add-k Bigram k=0.1: 0.016561097915017156
Add-k Bigram k=0.5: 0.005214723926380368
Add-k Bigram k=0.7: 0.003907437815483792

Sentence: tom sawyer went home
Backoff Probability: 1.8929810278376783e-10
